In [2]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "153Q3cVfJcgOBRtxV7Q2Me6877FTlWfxy5o0BmWKs9v4"
SHEET_NAME = "Rostock clients"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [17]:
duck.sql(f"""
    SELECT row_number() over(), *
    FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
    )

         """)

┌──────────────────────┬─────────────┬────────────────────┬───────────────────┬────────────────────┬─────────┐
│ row_number() OVER () │ easybill_id │      zoho_id       │ WZ_Code from zoho │ is_valid_for_padoa │   neu   │
│        int64         │   varchar   │      varchar       │      varchar      │      varchar       │ varchar │
├──────────────────────┼─────────────┼────────────────────┼───────────────────┼────────────────────┼─────────┤
│                    1 │ 130001582   │ 386758000010035065 │ 28                │ 28                 │ 28.99.9 │
│                    2 │ 130000853   │ 386758000035074001 │ 25.11.0           │ 25.11.0            │ 25.11.0 │
│                    3 │ 130002049   │ 386758000056063200 │ 08.12.0           │ #N/A               │ 08.11.0 │
│                    4 │ 130002003   │ 386758000056063200 │ 08.12.0           │ #N/A               │ 08.11.0 │
│                    5 │ 130000660   │ 386758000030503130 │ 23.1              │ #N/A               │ 23.15.0 │
│

In [3]:

duck.sql(
    f"""
    create or replace table wz as 
    SELECT *
    FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
    )
"""
)

In [13]:
duck.sql("select * from wz")

┌─────────────┬────────────────────┬───────────────────┬────────────────────┬─────────┐
│ easybill_id │      zoho_id       │ WZ_Code from zoho │ is_valid_for_padoa │   neu   │
│   varchar   │      varchar       │      varchar      │      varchar       │ varchar │
├─────────────┼────────────────────┼───────────────────┼────────────────────┼─────────┤
│ 130001582   │ 386758000010035065 │ 28                │ 28                 │ 28.99.9 │
│ 130000853   │ 386758000035074001 │ 25.11.0           │ 25.11.0            │ 25.11.0 │
│ 130002049   │ 386758000056063200 │ 08.12.0           │ #N/A               │ 08.11.0 │
│ 130002003   │ 386758000056063200 │ 08.12.0           │ #N/A               │ 08.11.0 │
│ 130000660   │ 386758000030503130 │ 23.1              │ #N/A               │ 23.15.0 │
│ 130002331   │ 386758000055210823 │ 25.1              │ #N/A               │ 25.11.0 │
│ 130001576   │ 386758000010035495 │ 43.21.0           │ 43.21.0            │ NULL    │
│ 130000189   │ 3867580000100347

In [18]:
duck.sql(
    """
    update pg.zoho.Accounts a
    set "WZ_Code" = coalesce(neu, is_valid_for_padoa)
    from wz 
    where a.Id = wz.zoho_id;
    """
)

In [ ]:
")